# DDoS Traffic Classification — LSTM + Attention

**Multiclass DDoS detekcija koriscenjem LSTM-a sa attention mehanizmom i TimeSeriesSplit kros-validacijom.**

### Klase:
- `normal`, `udp_flood_large`, `dns_amplification`, `subnet_carpet_bombing`
- `syn_flood`, `icmp_flood`, `udp_flood_mixed`, `ntp_amplification`, `ack_flood`

---

## 1. Instalacija zavisnosti

In [ ]:
# Instalacija potrebnih biblioteka (vecina je vec dostupna u Colab-u)
!pip install -q torch torchvision scikit-learn matplotlib seaborn pandas numpy

## 2. Google Drive (opcionalno)

In [ ]:
# Montiranje Google Drive-a — preskoci ako koristis direktan upload
from google.colab import drive
drive.mount('/content/drive')

# Ispravi ove putanje prema tvojoj strukturi na Drive-u:
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ddos_project'

import sys
sys.path.insert(0, DRIVE_PROJECT_PATH)

## 3. Upload CSV dataseta

### 3a. functions.py

In [ ]:
%%writefile functions.py
import random
import math
import csv
import torch
import sys
import numpy as np

from collections import defaultdict
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# Featurei koji su u opsegu 0-1
RATIO_FEATURES = {
    "udp_ratio", "tcp_ratio", "icmp_ratio",
    "tcp_syn_ratio", "tcp_ack_ratio", "tcp_fin_ratio",
    "dns_query_ratio", "dns_response_ratio",
    "top_src_ip_packet_share", "top_src_ip_byte_share",
    "top_dst_port_share", "dst_subnet_spread",
}

# Featurei koji moraju biti pozitivni
POSITIVE_FEATURES = {
    "packet_rate", "byte_rate", "avg_packet_size", "std_packet_size",
    "unique_src_ips", "unique_dst_ips", "unique_flows",
    "src_ip_entropy", "dst_ip_entropy", "dst_port_entropy",
}

def rand_uniform(min_val: float, max_val: float) -> float:
    return min_val + random.random() * (max_val - min_val)

# Box-Muller transformation
def rand_normal(mean: float, std: float) -> float:
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mean + std * z

def clamp(x: float, low: float, high: float) -> float:
    return max(low, min(high, x))

def write_csv(dataset: list[dict], filename: str) -> None:
    if not dataset:
        print("Dataset is empty, nothing to write.")
        return

    if not filename.endswith(".csv"):
        filename += ".csv"

    fieldnames = list(dataset[0].keys())

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(dataset)

    print(f"Written {len(dataset)} rows to '{filename}'.")

def format_timestamp(ms: int) -> str:
    try:
        # Konvertuje milisekunde u formatirani string
        dt = datetime.fromtimestamp(ms / 1000)
        dt_final = dt.replace(tzinfo=ZoneInfo('UTC'))
        return dt_final.strftime("%Y-%m-%dT%H:%M:%S")
    except Exception as e:
        print(f'Exception functions | format_timestamp: {e} Line: {sys.exc_info()[2].tb_lineno}')


def add_window_metadata(sample: dict, window_id: int, timestamp: int, active_atk: int) -> dict:
    # F-je koje dodaju vremenske serije podacima
    return {
        **sample,
        "window_id": window_id,
        "timestamp": timestamp,
        "ts_formated": format_timestamp(timestamp),
        "attack_active": int(active_atk),
    }


def compute_class_weights(dataset: list[dict], labels: list[str]) -> dict:
    """
    Racuna inverse-frequency weight po klasi.
    Vraca recnik {label: weight} koji se prosledjuje CrossEntropyLoss-u.
    """
    try:
        counts = {label: 0 for label in labels}
        for sample in dataset:
            lbl = sample.get("label")
            if lbl in counts:
                counts[lbl] += 1

        total = sum(counts.values())
        n_classes = len(labels)

        weights = {}
        for label, count in counts.items():
            weights[label] = total / (n_classes * count) if count > 0 else 1.0

        print("\nClass weights:")
        for label, w in weights.items():
            print(f"  {label:<25} count: {counts[label]:>7}   weight: {w:.4f}")

        return weights

    except Exception as e:
        print(f'Exception functions | compute_class_weights: {e} Line: {sys.exc_info()[2].tb_lineno}')


def get_class_weights_tensor(dataset: list[dict], labels: list[str], device) -> "torch.Tensor":
    try:
        weights_dict = compute_class_weights(dataset, labels)
        weights_list = [weights_dict.get(label, 1.0) for label in labels]
        return torch.tensor(weights_list, dtype=torch.float32).to(device)
    except Exception as e:
        print(f'Exception functions | get_class_weights_tensor: {e} Line: {sys.exc_info()[2].tb_lineno}')


def oversample_minority_classes(
    dataset: list[dict],
    labels: list[str],
    target_ratio: float = 0.5,
) -> list[dict]:
    """
    Oversampluje manjinske klase do target_ratio * count(normal).
    target_ratio=0.5 znaci da svaka napadna klasa ima bar 50% uzoraka normal klase.
    Koristi add_noise za male varijacije pri dupliranju uzoraka.
    """
    try:
        by_label = defaultdict(list)
        for sample in dataset:
            by_label[sample["label"]].append(sample)

        normal_count = len(by_label.get("normal", []))
        target_count = int(normal_count * target_ratio)

        print(f"\nOversampling — target by class: {target_count} (normal: {normal_count})")

        oversampled = list(dataset)

        for label in labels:
            if label == "normal":
                continue

            current = by_label.get(label, [])
            current_count = len(current)

            if current_count >= target_count:
                print(f"  {label:<25} {current_count:>7} skipping this")
                continue

            needed = target_count - current_count
            print(f"  {label:<25} {current_count:>7} > adding {needed} sample")

            for i in range(needed):
                base = random.choice(current)
                noisy = add_noise(base, noise_level=0.04)
                noisy["label"]         = label
                noisy["attack_active"] = base["attack_active"]
                noisy["window_id"]     = base["window_id"]
                noisy["timestamp"]     = base["timestamp"] + i
                noisy["ts_formated"]   = format_timestamp(noisy["timestamp"])
                oversampled.append(noisy)

        return oversampled

    except Exception as e:
        print(f'Exception functions | oversample_minority_classes: {e} Line: {sys.exc_info()[2].tb_lineno}')


def add_noise(sample: dict, noise_level: float = 0.05) -> dict:
    """
    Dodaje Gaussov sum na numericke featuere.
    noise_level = standardna devijacija kao procenat vrednosti featura.
    """
    try:
        noisy = dict(sample)
        for key, val in sample.items():
            if not isinstance(val, (int, float)):
                continue
            if key in ("window_id", "timestamp", "attack_active"):
                continue

            noise = np.random.normal(0, abs(val) * noise_level + 1e-6)
            noisy_val = val + noise

            if key in RATIO_FEATURES:
                noisy_val = float(np.clip(noisy_val, 0.0, 1.0))
            elif key in POSITIVE_FEATURES:
                noisy_val = max(0.0, noisy_val)

            noisy[key] = noisy_val

        return noisy

    except Exception as e:
        print(f'Exception functions | add_noise: {e} Line: {sys.exc_info()[2].tb_lineno}')


def blend_samples(sample_a: dict, sample_b: dict, alpha: float) -> dict:
    """
    Linearno interpoluje izmedju dva uzorka.
    alpha=0.0 -> sample_a, alpha=1.0 -> sample_b
    Koristi se za tranzicione periode.
    """
    try:
        blended = dict(sample_a)
        for key, val_a in sample_a.items():
            if not isinstance(val_a, (int, float)):
                continue
            if key in ("window_id", "timestamp", "attack_active", "label"):
                continue
            val_b = sample_b.get(key, val_a)
            blended[key] = val_a * (1 - alpha) + val_b * alpha
        return blended
    except Exception as e:
        print(f'Exception functions | blend_samples: {e} Line: {sys.exc_info()[2].tb_lineno}')


def generate_transition(
    from_fn, to_fn,
    from_label: str, to_label: str,
    n_windows: int, start_ts: int, window_ms: int,
) -> list[dict]:
    """
    Generise tranzicioni period izmedju dva tipa saobracaja.
    Prvih 50% prozora: label from_label sa rastucim alpha ka to_fn
    Drugih 50% prozora: label to_label sa opadajucim alpha
    """
    try:
        result = []
        for i in range(n_windows):
            alpha  = i / n_windows
            base_a = from_fn()
            base_b = to_fn()

            blended = blend_samples(base_a, base_b, alpha)
            blended = add_noise(blended, noise_level=0.08)

            label = from_label if alpha < 0.5 else to_label
            ts    = start_ts + i * window_ms

            result.append({
                **blended,
                "label":          label,
                "window_id":      i,
                "timestamp":      ts,
                "ts_formated":    format_timestamp(ts),
                "attack_active":  1 if label != "normal" else 0,
            })

        return result

    except Exception as e:
        print(f'Exception functions | generate_transition: {e} Line: {sys.exc_info()[2].tb_lineno}')

### 3b. metrics_exporter.py

In [ ]:
%%writefile metrics_exporter.py
import json
import logging
import sys
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    roc_auc_score,
)

logger = logging.getLogger(__name__)


def export_metrics_to_json(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
    class_labels: list[str],
    output_dir: str = ".",
    filename: str = "eval_metrics.json",
) -> str:
    """
    Racuna sve metrike evaluacije i cuva ih u JSON fajl.

    Parametri:
        y_true       — stvarne oznake (integer indeksi klasa)
        y_pred       — predvidjene oznake (integer indeksi klasa)
        y_proba      — verovatnoca po klasi (shape: n_samples x n_classes)
        class_labels — lista naziva klasa u ispravnom redosledu
        output_dir   — direktorijum za cuvanje JSON fajla
        filename     — naziv izlaznog fajla

    Vraca:
        Apsolutnu putanju do sacuvanog JSON fajla.
    """
    try:
        report = classification_report(
            y_true, y_pred,
            target_names=class_labels,
            output_dict=True,
            zero_division=0,
        )
        # Cuvamo samo per-class stavke (iskljucujemo accuracy, macro avg, weighted avg)
        per_class_report = {
            cls: metrics
            for cls, metrics in report.items()
            if isinstance(metrics, dict)
        }

        cm  = confusion_matrix(y_true, y_pred)
        mcc = float(matthews_corrcoef(y_true, y_pred))
        roc_auc = _compute_roc_auc_per_class(y_true, y_proba, class_labels)

        metrics_payload = {
            "timestamp":             datetime.now().isoformat(),
            "class_labels":          class_labels,
            "mcc_score":             mcc,
            "classification_report": per_class_report,
            "confusion_matrix":      cm.tolist(),
            "roc_auc_scores":        roc_auc,
            "summary": {
                "macro_f1":        report.get("macro avg",    {}).get("f1-score",  0.0),
                "weighted_f1":     report.get("weighted avg", {}).get("f1-score",  0.0),
                "macro_precision": report.get("macro avg",    {}).get("precision", 0.0),
                "macro_recall":    report.get("macro avg",    {}).get("recall",    0.0),
                "total_samples":   int(report.get("macro avg", {}).get("support",  0)),
            },
        }

        output_path = Path(output_dir) / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(metrics_payload, f, indent=2)

        logger.info(f"Metrics saved to: {output_path}")
        return str(output_path.resolve())

    except Exception as e:
        logger.error(f"Failed to export metrics: {e} | Line: {sys.exc_info()[2].tb_lineno}")
        raise


def _compute_roc_auc_per_class(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    class_labels: list[str],
) -> dict:
    """
    Racuna ROC-AUC za svaku klasu metodom one-vs-rest.
    Vraca 0.0 za klasu koja nije prisutna u y_true (ne moze se izracunati).
    """
    roc_auc: dict = {}

    for i, label in enumerate(class_labels):
        if i >= y_proba.shape[1]:
            roc_auc[label] = 0.0
            continue

        y_true_binary = (y_true == i).astype(int)

        if y_true_binary.sum() == 0:
            logger.warning(f"Class '{label}' not present in y_true — ROC-AUC set to 0.0")
            roc_auc[label] = 0.0
            continue

        try:
            roc_auc[label] = float(roc_auc_score(y_true_binary, y_proba[:, i]))
        except Exception as e:
            logger.warning(f"ROC-AUC failed for class '{label}': {e}")
            roc_auc[label] = 0.0

    return roc_auc


def load_metrics_from_json(path: str) -> Optional[dict]:
    """
    Ucitava prethodno sacuvane metrike iz JSON fajla.
    Vraca None ako fajl ne postoji ili je ostecen.
    """
    p = Path(path)
    if not p.exists():
        logger.warning(f"Metrics file not found: {path}")
        return None

    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        logger.error(f"Failed to load metrics from '{path}': {e}")
        return None

### 3c. Upload CSV dataseta

In [ ]:
from google.colab import files
import os

print("Upload: CSV dataset")
uploaded = files.upload()

# Provera sta je ucitano
for name in uploaded:
    print(f"  Uploaded: {name}  ({len(uploaded[name])} bytes)")

# Putanja do CSV-a (ispravi naziv ako se razlikuje)
CSV_NAMES = [f for f in uploaded if f.endswith('.csv')]
if CSV_NAMES:
    CSV_PATH = CSV_NAMES[0]
    print(f"\nCSV dataset: {CSV_PATH}")
else:
    print("\n[!] Nije uploadovan CSV fajl. Ispravi CSV_PATH rucno.")
    CSV_PATH = 'your_dataset.csv'  # <-- promeni rucno ako je potrebno

## 4. Importi i konstante

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    matthews_corrcoef,
)
from sklearn.preprocessing import label_binarize
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

from functions import *
from metrics_exporter import export_metrics_to_json

print(f"PyTorch verzija: {torch.__version__}")
print(f"CUDA dostupan:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

In [ ]:
# Hiperpametri 
SEQUENCE_LEN  = 50
BATCH_SIZE    = 64
HIDDEN_SIZE   = 128
NUM_LAYERS    = 2
DROPOUT       = 0.3
LEARNING_RATE = 1e-3
EPOCHS        = 20
N_SPLITS      = 5   # Broj foldova za TimeSeriesSplit

# Izlazni direktorijum za grafove i rezultate
OUTPUT_DIR  = '/content/ddos_output'
GRAPHS_DIR  = os.path.join(OUTPUT_DIR, 'graphs')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
MODEL_SAVE  = os.path.join(OUTPUT_DIR, 'ddos_lstm_attention.pt')

os.makedirs(GRAPHS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Grafovi -> {GRAPHS_DIR}")
print(f"Rezultati -> {RESULTS_DIR}")

# Kolone koje se iskljucuju iz feature seta
EXCLUDED_COLS = [
    'label', 'window_id', 'timestamp', 'ts_formated',
    'attack_active', 'instance_id', 'vector_id',
]

#Sve klase, redosled mora biti konzistentan sa LabelEncoder-om 
LABELS = [
    'normal',
    'udp_flood_large',
    'dns_amplification',
    'subnet_carpet_bombing',
    'syn_flood',
    'icmp_flood',
    'udp_flood_mixed',
    'ntp_amplification',
    'ack_flood',
]

## 5. Dataset klasa

In [ ]:
class DDoSDataset(Dataset):
    def __init__(self, sequences: np.ndarray, labels: np.ndarray):
        self.X = torch.tensor(sequences, dtype=torch.float32)
        self.Y = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

## 6. Modeli — Attention sloj, DDoSLSTM, DDoSLSTMAttention

In [ ]:
class AttentionLayer(nn.Module):
    """Nauceni vektor paznje koji ocenjuje svaki timestep."""

    def __init__(self, hidden_size: int):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor) -> tuple:
        """
        lstm_out: (batch, seq_len, hidden_size)
        Vraca:
            context : (batch, hidden_size) — tezinski zbir svih timestepova
            weights : (batch, seq_len)     — tezine paznje po timestepu
        """
        scores  = self.attention(lstm_out).squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(scores, dim=1)              # (batch, seq_len)
        context = torch.bmm(
            weights.unsqueeze(1), lstm_out              # (batch,1,seq_len) x (batch,seq_len,hidden)
        ).squeeze(1)                                    # (batch, hidden_size)
        return context, weights

In [ ]:
class DDoSLSTM(nn.Module):
    """Standardni LSTM classifier bez attention-a."""

    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        try:
            lstm_out, _ = self.lstm(x)
            last_hidden = lstm_out[:, -1, :]
            return self.classifier(last_hidden)
        except Exception as e:
            print(f'Exception | DDoSLSTM.forward: {e}  Line: {sys.exc_info()[2].tb_lineno}')


class DDoSLSTMAttention(nn.Module):
    """LSTM classifier sa attention mehanizmom."""

    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention   = AttentionLayer(hidden_size)
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> tuple:
        """
        Vraca logits i attention weights (korisno za vizualizaciju).
        """
        try:
            lstm_out, _      = self.lstm(x)
            context, weights = self.attention(lstm_out)
            logits           = self.classifier(context)
            return logits, weights
        except Exception as e:
            print(f'Exception | DDoSLSTMAttention.forward: {e}  Line: {sys.exc_info()[2].tb_lineno}')

## 7. Priprema podataka

In [ ]:
def prepare_data(csv_path: str, seq_len: int = SEQUENCE_LEN):
    """
    Ucitava CSV, sortira po timestamp-u, skalira featuere i pravi sekvence.
    Vraca sekvence, labele, scaler, label encoder i listu feature kolona.
    """
    try:
        df = pd.read_csv(csv_path)

        if 'timestamp' in df.columns:
            df = df.sort_values('timestamp').reset_index(drop=True)

        feature_cols = [c for c in df.columns if c not in EXCLUDED_COLS]
        X_raw        = df[feature_cols].values.astype(np.float32)

        label_enc         = LabelEncoder()
        label_enc.classes_ = np.array(LABELS)
        Y_raw             = label_enc.transform(df['label'].values)

        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)

        sequences, labels = [], []
        for i in range(len(X_scaled) - seq_len):
            sequences.append(X_scaled[i: i + seq_len])
            labels.append(Y_raw[i + seq_len - 1])

        print(f"Dataset ucitan: {len(df):,} redova  ->  {len(sequences):,} sekvenci")
        print(f"Feature kolona: {len(feature_cols)}")

        return (
            np.array(sequences),
            np.array(labels),
            scaler,
            label_enc,
            feature_cols,
        )

    except Exception as e:
        print(f'Exception | prepare_data: {e}  Line: {sys.exc_info()[2].tb_lineno}')


def make_dataloaders(csv_path: str, seq_len: int = SEQUENCE_LEN, batch_size: int = BATCH_SIZE):
    """Pravi DataLoader-e za train/val/test split (60/20/20)."""
    try:
        sequences, labels, scaler, le, feature_cols = prepare_data(csv_path, seq_len)

        n         = len(sequences)
        train_end = int(n * 0.60)
        val_end   = int(n * 0.80)

        X_train, y_train = sequences[:train_end],        labels[:train_end]
        X_val,   y_val   = sequences[train_end:val_end], labels[train_end:val_end]
        X_test,  y_test  = sequences[val_end:],          labels[val_end:]

        print(f"Train:      {len(X_train):>7,} samples")
        print(f"Validation: {len(X_val):>7,} samples")
        print(f"Test:       {len(X_test):>7,} samples")

        train_loader = DataLoader(DDoSDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(DDoSDataset(X_val,   y_val),   batch_size=batch_size, shuffle=False)
        test_loader  = DataLoader(DDoSDataset(X_test,  y_test),  batch_size=batch_size, shuffle=False)

        return train_loader, val_loader, test_loader, scaler, le, len(feature_cols)

    except Exception as e:
        print(f'Exception | make_dataloaders: {e}  Line: {sys.exc_info()[2].tb_lineno}')

## 8. Training funkcije

In [ ]:
def singular_epoch(model, loader, optimizer, criterion, device):
    """Jedan trening epoch za DDoSLSTM (bez attention)."""
    try:
        model.train()
        total_loss, correct = 0.0, 0

        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)
            correct    += (logits.argmax(dim=1) == y_batch).sum().item()

        n = len(loader.dataset)
        return total_loss / n, correct / n

    except Exception as e:
        print(f'Exception | singular_epoch: {e}  Line: {sys.exc_info()[2].tb_lineno}')


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Brza evaluacija — samo loss i accuracy."""
    try:
        model.eval()
        total_loss, correct = 0.0, 0

        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            correct    += (logits.argmax(dim=1) == y_batch).sum().item()

        n = len(loader.dataset)
        return total_loss / n, correct / n

    except Exception as e:
        print(f'Exception | evaluate: {e}  Line: {sys.exc_info()[2].tb_lineno}')


def train_fold(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, fold: int,
) -> dict:
    """Trenira model za jedan fold i vraca metriku najboljeg epocha."""
    best_val_loss = float('inf')
    best_metrics  = {}

    for epoch in range(1, EPOCHS + 1):
        # ── Trening ───────────────────────────────────────────────────────────
        model.train()
        train_loss, train_correct = 0.0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits, _ = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss    += loss.item() * len(y_batch)
            train_correct += (logits.argmax(dim=1) == y_batch).sum().item()

        # ── Validacija ────────────────────────────────────────────────────────
        model.eval()
        val_loss, val_correct = 0.0, 0
        all_preds, all_targets, all_probs = [], [], []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits, _ = model(X_batch)
                loss      = criterion(logits, y_batch)
                val_loss    += loss.item() * len(y_batch)
                val_correct += (logits.argmax(dim=1) == y_batch).sum().item()
                all_preds.extend(logits.argmax(dim=1).cpu().numpy())
                all_targets.extend(y_batch.cpu().numpy())
                all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())

        n_train = len(train_loader.dataset)
        n_val   = len(val_loader.dataset)
        t_loss  = train_loss / n_train
        t_acc   = train_correct / n_train
        v_loss  = val_loss / n_val
        v_acc   = val_correct / n_val

        scheduler.step(v_loss)

        print(
            f"  Fold {fold} | Epoch {epoch:>2}/{EPOCHS} | "
            f"Train loss: {t_loss:.4f}  acc: {t_acc:.3f} | "
            f"Val loss: {v_loss:.4f}  acc: {v_acc:.3f}"
        )

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_metrics  = {
                'fold':    fold,
                'val_loss': v_loss,
                'val_acc':  v_acc,
                'mcc':     matthews_corrcoef(all_targets, all_preds),
                'preds':   all_preds,
                'targets': all_targets,
                'probs':   np.array(all_probs),
            }

    return best_metrics

## 9. Evaluacija i plotovanje

In [ ]:
@torch.no_grad()
def evaluate_full(model, loader, criterion, device, label_names: list):
    """
    Kompletna evaluacija:
    - Loss i accuracy
    - Confusion matrix
    - Classification report (precision, recall, F1)
    - ROC-AUC (one-vs-rest)
    - Matthews Correlation Coefficient
    """
    try:
        model.eval()
        total_loss = 0.0
        all_preds, all_targets, all_probs = [], [], []

        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            probs  = torch.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

        all_preds   = np.array(all_preds)
        all_targets = np.array(all_targets)
        all_probs   = np.array(all_probs)

        avg_loss = total_loss / len(loader.dataset)
        accuracy = (all_preds == all_targets).mean()

        print('=' * 65)
        print(f'  Loss: {avg_loss:.4f}   Accuracy: {accuracy:.4f}')
        print('=' * 65)

        print('\nClassification Report:')
        print(classification_report(
            all_targets, all_preds,
            labels=list(range(len(label_names))),
            target_names=label_names,
            digits=4,
        ))

        mcc = matthews_corrcoef(all_targets, all_preds)
        print(f'Matthews Correlation Coefficient (MCC): {mcc:.4f}\n')

        y_bin = label_binarize(all_targets, classes=list(range(len(label_names))))
        try:
            roc_auc = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')
            print(f'ROC-AUC (macro, one-vs-rest): {roc_auc:.4f}\n')
        except ValueError as ve:
            print(f'ROC-AUC nije mogao biti izracunat: {ve}\n')

        plot_confusion_matrix(all_targets, all_preds, label_names)
        plot_roc_curves(y_bin, all_probs, label_names)

        return {
            'loss':     avg_loss,
            'accuracy': accuracy,
            'mcc':      mcc,
            'preds':    all_preds,
            'targets':  all_targets,
            'probs':    all_probs,
        }

    except Exception as e:
        print(f'Exception | evaluate_full: {e}  Line: {sys.exc_info()[2].tb_lineno}')


def plot_confusion_matrix(y_true, y_pred, label_names: list):
    """Plotuje apsolutnu i normalizovanu confusion matricu."""
    try:
        cm      = confusion_matrix(y_true, y_pred)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        fig, axes = plt.subplots(1, 2, figsize=(20, 8))

        for ax, data, title, fmt in zip(
            axes,
            [cm, cm_norm],
            ['Confusion Matrix (absolute numbers)', 'Confusion Matrix (normalized)'],
            ['d', '.2f'],
        ):
            sns.heatmap(
                data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                ax=ax, linewidths=0.5,
            )
            ax.set_title(title, fontsize=13, pad=12)
            ax.set_xlabel('Predicted', fontsize=11)
            ax.set_ylabel('Actual',    fontsize=11)
            ax.tick_params(axis='x', rotation=45)
            ax.tick_params(axis='y', rotation=0)

        plt.tight_layout()
        save_path = os.path.join(GRAPHS_DIR, 'confusion_matrix.png')
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f'Saved confusion matrix -> {save_path}')

    except Exception as e:
        print(f'Exception | plot_confusion_matrix: {e}  Line: {sys.exc_info()[2].tb_lineno}')


def plot_roc_curves(y_bin, all_probs, label_names: list):
    """Plotuje ROC krive po klasi (one-vs-rest)."""
    try:
        n_classes = len(label_names)
        colors    = plt.cm.tab10(np.linspace(0, 1, n_classes))

        plt.figure(figsize=(10, 7))

        for i, (name, color) in enumerate(zip(label_names, colors)):
            fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
            auc         = roc_auc_score(y_bin[:, i], all_probs[:, i])
            plt.plot(fpr, tpr, color=color, lw=1.8, label=f'{name}  (AUC = {auc:.3f})')

        plt.plot([0, 1], [0, 1], 'k--', lw=1)
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate',  fontsize=12)
        plt.title('ROC curves by class (one-vs-rest)', fontsize=13)
        plt.legend(loc='lower right', fontsize=9)
        plt.tight_layout()
        save_path = os.path.join(GRAPHS_DIR, 'roc_curves.png')
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f'Saved ROC curves -> {save_path}')

    except Exception as e:
        print(f'Exception | plot_roc_curves: {e}  Line: {sys.exc_info()[2].tb_lineno}')

## 10. Cross-validacija — glavna funkcija

In [ ]:
def cross_validate(csv_path: str, save_path: str = MODEL_SAVE):
    """
    TimeSeriesSplit cross-validacija sa LSTM + Attention modelom.
    Svaki fold cuva hronoloski redosled — nema data leakage-a.
    Na kraju ispisuje prosecnu metriku kroz sve foldove.
    """
    try:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using: {device}')

        sequences, labels, scaler, le, feature_cols = prepare_data(csv_path)
        num_features = len(feature_cols)
        num_classes  = len(le.classes_)

        dataset = DDoSDataset(sequences, labels)
        tss     = TimeSeriesSplit(n_splits=N_SPLITS)

        fold_metrics      = []
        best_overall_loss = float('inf')
        best_model_state  = None

        for fold, (train_idx, val_idx) in enumerate(tss.split(sequences), start=1):
            print(f"\n{'='*60}")
            print(f"  Fold {fold}/{N_SPLITS} | "
                  f"Train: {len(train_idx):,}  Val: {len(val_idx):,}")
            print(f"{'='*60}")

            train_loader = DataLoader(
                Subset(dataset, train_idx),
                batch_size=BATCH_SIZE, shuffle=False,
            )
            val_loader = DataLoader(
                Subset(dataset, val_idx),
                batch_size=BATCH_SIZE, shuffle=False,
            )

            model = DDoSLSTMAttention(
                input_size=num_features,
                hidden_size=HIDDEN_SIZE,
                num_layers=NUM_LAYERS,
                num_classes=num_classes,
                dropout=DROPOUT,
            ).to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, patience=3, factor=0.5
            )
            criterion = nn.CrossEntropyLoss()

            metrics = train_fold(
                model, train_loader, val_loader,
                criterion, optimizer, scheduler,
                device, fold,
            )
            fold_metrics.append(metrics)

            if metrics['val_loss'] < best_overall_loss:
                best_overall_loss = metrics['val_loss']
                best_model_state  = model.state_dict()
                torch.save({
                    'model_state':   best_model_state,
                    'scaler':        scaler,
                    'label_encoder': le,
                    'num_features':  num_features,
                    'architecture':  'LSTM+Attention',
                    'hyperparams': {
                        'hidden_size': HIDDEN_SIZE,
                        'num_layers':  NUM_LAYERS,
                        'dropout':     DROPOUT,
                        'seq_len':     SEQUENCE_LEN,
                    },
                }, save_path)
                print(f"  New best model saved (fold {fold}) -> {save_path}")

        # ── Sumarni rezultati ─────────────────────────────────────────────────
        print(f"\n{'='*60}")
        print('  Cross-validation summary')
        print(f"{'='*60}")
        print(f"  {'Fold':<8} {'Val Loss':<12} {'Val Acc':<12} {'MCC'}")
        print(f"  {'-'*48}")

        for m in fold_metrics:
            print(f"  {m['fold']:<8} {m['val_loss']:<12.4f} "
                  f"{m['val_acc']:<12.4f} {m['mcc']:.4f}")

        avg_loss = np.mean([m['val_loss'] for m in fold_metrics])
        avg_acc  = np.mean([m['val_acc']  for m in fold_metrics])
        avg_mcc  = np.mean([m['mcc']      for m in fold_metrics])
        std_acc  = np.std( [m['val_acc']  for m in fold_metrics])

        print(f"  {'-'*48}")
        print(f"  {'Avg':<8} {avg_loss:<12.4f} {avg_acc:<12.4f} {avg_mcc:.4f}")
        print(f"  {'Std':<8} {'':12} {std_acc:<12.4f}")
        print(f"{'='*60}\n")

        # ── Classification report najboljeg folda ─────────────────────────────
        best_fold = min(fold_metrics, key=lambda m: m['val_loss'])
        print(f"Classification report (best fold {best_fold['fold']}):")
        print(classification_report(
            best_fold['targets'], best_fold['preds'],
            labels=list(range(num_classes)),
            target_names=LABELS,
            digits=4,
            zero_division=0,
        ))

        best_targets = np.array(best_fold['targets'])
        best_preds   = np.array(best_fold['preds'])
        best_probs   = best_fold['probs']

        print(f"MCC (best fold {best_fold['fold']}): {best_fold['mcc']:.4f}\n")

        plot_confusion_matrix(best_targets, best_preds, LABELS)

        y_bin = label_binarize(best_targets, classes=list(range(num_classes)))
        plot_roc_curves(y_bin, best_probs, LABELS)

        # ── Export metrika u JSON za LangGraph analizu ────────────────────────
        print('Exporting evaluation metrics to JSON...')
        json_path = export_metrics_to_json(
            y_true=best_targets,
            y_pred=best_preds,
            y_proba=best_probs,
            class_labels=LABELS,
            output_dir=RESULTS_DIR,
            filename='eval_metrics.json',
        )
        print(f'Metrics saved -> {json_path}')

        print('\nTraining finished!')
        return fold_metrics

    except Exception as e:
        print(f'Exception | cross_validate: {e}  Line: {sys.exc_info()[2].tb_lineno}')

## 11. Pokretanje treninga

> Proveri da je `CSV_PATH` ispravno postavljen u celiji za upload (korak 3).

In [ ]:
# Pokretanje cross-validacije
# CSV_PATH je automatski postavljen u celiji za upload;
# ovde ga mozes rucno pregaziti ako je potrebno:
# CSV_PATH = '/content/drive/MyDrive/ddos_project/dataset.csv'

fold_results = cross_validate(CSV_PATH, save_path=MODEL_SAVE)

## 12. Preuzimanje rezultata

In [ ]:
from google.colab import files
import glob

# Preuzimanje sacuvanog modela
files.download(MODEL_SAVE)

# Preuzimanje grafova
for png in glob.glob(os.path.join(GRAPHS_DIR, '*.png')):
    files.download(png)

# Preuzimanje JSON metrika
json_metrics = os.path.join(RESULTS_DIR, 'eval_metrics.json')
if os.path.exists(json_metrics):
    files.download(json_metrics)